In [33]:
from enum import Enum
from dataclasses import dataclass
 
class Dir(Enum):
    Buy = 1
    Sell = -1

Price = int

@dataclass(frozen=True, kw_only=True)
class Order:
    auction_id: int
    price: Price | None
    size: int
    dir: Dir

    def get_price(self) -> Price:
        LARGE_NUMBER = 1000000
        if self.price is None:
            return 0 if self.dir == Dir.Sell else LARGE_NUMBER
        else:
            return self.price
    

def sort_orders(orders: list):
    # returns a tuple:
    # first: all orders with price None (market orders)
    # second: Other orders with price sorted in ascending order
    market_orders = [o for o in orders if o.price is None]
    limit_orders = [o for o in orders if o.price is not None]
    limit_orders.sort(key=lambda o: o.price)
    return market_orders, limit_orders


def find_fill_price(bids: list, asks: list) -> Price | None:
    bid_market, bid_limit = sort_orders(bids)
    ask_market, ask_limit = sort_orders(asks)
    sorted_bids = bid_limit + bid_market
    sorted_asks = ask_market + ask_limit

    current_bid = 0
    current_ask = 0
    current_bid_price = 1000000
    current_ask_price = 0
    bid_size_left = 0
    ask_size_left = 0
    total_size_filled = 0

    # Step 1: Compute total_size_filled
    while (current_bid < len(sorted_bids) or bid_size_left > 0) and (current_ask < len(sorted_asks) or ask_size_left > 0):
        # print(f"Current bid: {current_bid}, Current ask: {current_ask}")
        if bid_size_left == 0:
            current_bid_price = sorted_bids[current_bid].get_price()
            while current_bid < len(sorted_bids):
                if sorted_bids[current_bid].get_price() != current_bid_price:
                    break
                bid_size_left += sorted_bids[current_bid].size
                current_bid += 1

        if ask_size_left == 0:
            current_ask_price = sorted_asks[current_ask].get_price()
            while current_ask < len(sorted_asks):
                if sorted_asks[current_ask].get_price() != current_ask_price:
                    break
                ask_size_left += sorted_asks[current_ask].size
                current_ask += 1

        # print(f"Current bid price: {current_bid_price}, Current ask price: {current_ask_price}")

        if current_bid_price >= current_ask_price:
            match_size = min(bid_size_left, ask_size_left)
            total_size_filled += match_size
            bid_size_left -= match_size
            ask_size_left -= match_size
        else:
            # Current bids cannot be filled
            bid_size_left = 0

        # print(f"Bid size left: {bid_size_left}, Ask size left: {ask_size_left}")

    # Step 2: Match orders
    if total_size_filled == 0:
        print("No fills")
        min_bid_price = sorted_asks[0].get_price() 
        max_bid_price = sorted_bids[-1].get_price()
        min_ask_price = sorted_bids[0].get_price()
        max_ask_price = sorted_asks[-1].get_price()
    else:
        filled_bids = []
        cur_i = len(sorted_bids) - 1
        cur_size = 0
        while cur_i >= 0 and cur_size < total_size_filled:
            filled_size = min(total_size_filled - cur_size, sorted_bids[cur_i].size)
            filled_bids.append([sorted_bids[cur_i].get_price(), filled_size])
            cur_size += filled_size
            cur_i -= 1
        filled_bids.reverse()
        filled_asks = []
        cur_i = 0
        cur_size = 0
        while cur_i < len(sorted_asks) and cur_size < total_size_filled:
            filled_size = min(total_size_filled - cur_size, sorted_asks[cur_i].size)
            filled_asks.append([sorted_asks[cur_i].get_price(), filled_size])
            cur_size += filled_size
            cur_i += 1
        current_bid = 0
        current_ask = 0
        current_bid_price = 1000000
        current_ask_price = 0
        bid_size_left = 0
        ask_size_left = 0
        print(filled_bids)
        print(filled_asks)
        min_bid_price = filled_bids[0][0]
        if bid_limit:
            max_bid_price = bid_limit[-1].get_price() + 1
        else:
            max_bid_price = min_bid_price 
        max_ask_price = filled_asks[-1][0] + 1
        if ask_limit:
            min_ask_price = ask_limit[0].get_price() - 1
        else:
            min_ask_price = max_ask_price

        while (current_bid < len(filled_bids) or bid_size_left > 0) and (current_ask < len(filled_asks) or ask_size_left > 0):
            if bid_size_left == 0:
                current_bid_price = filled_bids[current_bid][0]
                while current_bid < len(filled_bids):
                    if filled_bids[current_bid][0] != current_bid_price:
                        break
                    bid_size_left += filled_bids[current_bid][1]
                    current_bid += 1
                # print(f"Current bid price: {current_bid_price}, size left: {bid_size_left}")
            if ask_size_left == 0:
                current_ask_price = filled_asks[current_ask][0]
                while current_ask < len(filled_asks):
                    if filled_asks[current_ask][0] != current_ask_price:
                        break
                    ask_size_left += filled_asks[current_ask][1]
                    current_ask += 1
            assert current_bid_price >= current_ask_price, f"{current_bid_price} < {current_ask_price}"
            match_size = min(bid_size_left, ask_size_left)
            bid_size_left -= match_size
            ask_size_left -= match_size
            print(f"Matched at bid: {current_bid_price}, ask: {current_ask_price}, size: {match_size}")
    
    return min_bid_price, max_bid_price, min_ask_price, max_ask_price


bid_prices = [90, 91, 93, 95, 96, 97, 99, None]
bid_sizes = [40, 40, 20, 20, 15, 15, 10, 5]

ask_prices = [None, 95, 95, 98, 100, 101, 103, 104]
ask_sizes = [10, 5, 10, 15, 20, 30, 40, 40]

bids = [Order(auction_id=i, price=price, size=size, dir=Dir.Buy) for i, (price, size) in enumerate(zip(bid_prices, bid_sizes))]
asks = [Order(auction_id=i, price=price, size=size, dir=Dir.Sell) for i, (price, size) in enumerate(zip(ask_prices, ask_sizes))]
find_fill_price(bids, asks)


[[96, 10], [97, 15], [99, 10], [1000000, 5]]
[[0, 10], [95, 5], [95, 10], [98, 15]]
Matched at bid: 96, ask: 0, size: 10
Matched at bid: 97, ask: 95, size: 15
Matched at bid: 99, ask: 98, size: 10
Matched at bid: 1000000, ask: 98, size: 5


(96, 100, 94, 99)

In [35]:
import torch

encoder_layer = torch.nn.TransformerEncoderLayer(d_model=64, nhead=8, dim_feedforward=256, dropout=0.1, batch_first=True)
src = torch.rand(32, 10, 64) # (batch_size, seq_len, d_model)
encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers=3)
out = encoder(src)
print(out.shape)  # Should be (32, 10, 64)

torch.Size([32, 10, 64])
